# ResNet Fine-tuning for Radar Pose Estimation

This notebook fine-tunes a torchvision ResNet backbone on radar heatmaps to
predict human pose keypoints. It mirrors the main pipeline (dataset, losses,
metrics, AMP, early stopping) for a fair comparison.


In [ ]:
from pathlib import Path
from torch import serialization as torch_serialization
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import numpy as np
import os, sys, glob, math, random, numpy as np, cv2, torch, torchvision
import torch.amp as amp
import torch.nn as nn, torch.nn.functional as F


In [ ]:
# --- dataset constants
CAMERA_WIDTH = 640
CAMERA_HEIGHT = 480
LOG_SCALE_INPUT = True


In [ ]:
# --- data paths & runtime
BASE_DIR = Path.cwd()
DATA_DIR = Path(os.environ.get('RADAR_DATA_DIR', BASE_DIR / 'P1'))
if not DATA_DIR.exists():
    raise FileNotFoundError(f"Expected radar data folder at {DATA_DIR}. Set RADAR_DATA_DIR if stored elsewhere.")

OUT_DIR = BASE_DIR / 'checkpoint_resnet'
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = str(DATA_DIR)
OUT_DIR = str(OUT_DIR)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.backends.cudnn.benchmark = True

# allow loading numpy reconstruct in checkpoints when needed
reconstruct = getattr(np.core.multiarray, '_reconstruct', None)
if reconstruct is not None:
    torch_serialization.add_safe_globals([reconstruct])

TORCH_LOAD = torch.load
def load_checkpoint(path, map_location=None, weights_only=False):
    return TORCH_LOAD(path, map_location=map_location, weights_only=weights_only)

# be explicit with multiprocessing on Windows / notebooks
_start_method = None
if sys.platform == 'win32':
    _start_method = 'spawn'
if _start_method is not None:
    try:
        torch.multiprocessing.set_start_method(_start_method, force=True)
    except RuntimeError:
        pass

print('Using device:', device)


In [ ]:
# --- I/O & normalization utilities
def load_radar_frame(root, fid, out_res, log_scale=True):
    with np.load(os.path.join(root, f'{fid}_radar.npz')) as data:
        hm_hori = data['hm_hori'].astype(np.float32)
        hm_vert = data['hm_vert'].astype(np.float32)
    res_hori = cv2.resize(hm_hori, (out_res, out_res), interpolation=cv2.INTER_LINEAR)
    res_vert = cv2.resize(hm_vert, (out_res, out_res), interpolation=cv2.INTER_LINEAR)
    inp = np.stack([res_hori, res_vert], axis=0)
    if log_scale:
        inp = np.log1p(np.clip(inp, a_min=0.0, a_max=None))
    return inp

def compute_channel_stats(root, ids, out_res, cache_path=None, log_scale=True):
    if cache_path and os.path.exists(cache_path):
        cached = np.load(cache_path)
        mean, std = cached['mean'], cached['std']
        cached.close()
        return mean, std
    sum_c = np.zeros(2, dtype=np.float64)
    sumsq_c = np.zeros(2, dtype=np.float64)
    total_px = 0
    for fid in tqdm(ids, desc='stats', leave=False):
        frame = load_radar_frame(root, fid, out_res, log_scale=log_scale)
        reshaped = frame.reshape(2, -1)
        sum_c += reshaped.sum(axis=1)
        sumsq_c += (reshaped ** 2).sum(axis=1)
        total_px += frame.shape[1] * frame.shape[2]
    denom = max(total_px, 1)
    mean = sum_c / denom
    var = sumsq_c / denom - mean ** 2
    var = np.clip(var, 0.0, None)
    std = np.sqrt(var)
    mean = np.nan_to_num(mean.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    std = np.nan_to_num(std.astype(np.float32), nan=1.0, posinf=1.0, neginf=1.0)
    std = np.maximum(std, 1e-6)
    if cache_path:
        np.savez(cache_path, mean=mean, std=std)
    return mean, std


In [ ]:
# --- dataset
class RadarPoseDataset(Dataset):
    def __init__(self, data_dir, ids, out_res=128, sigma=2.0, mean=None, std=None, log_scale=True):
        self.root, self.ids, self.out_res, self.sigma = data_dir, ids, out_res, sigma
        self.mean = np.array(mean, dtype=np.float32) if mean is not None else None
        self.std = np.array(std, dtype=np.float32) if std is not None else None
        self.log_scale = log_scale
    def __len__(self):
        return len(self.ids)
    def _make_heatmaps(self, xy, vis):
        K, H, W = xy.shape[0], self.out_res, self.out_res
        yy, xx = np.meshgrid(np.arange(H), np.arange(W), indexing='ij')
        hms = np.zeros((K, H, W), np.float32)
        for k in range(K):
            if vis[k] < 0.5:
                continue
            x, y = xy[k]
            g = np.exp(-((yy - y) ** 2 + (xx - x) ** 2) / (2 * self.sigma ** 2))
            hms[k] = g
        return hms
    def __getitem__(self, i):
        fid = self.ids[i]
        inp = load_radar_frame(self.root, fid, self.out_res, log_scale=self.log_scale)
        if self.mean is not None and self.std is not None:
            inp = (inp - self.mean[:, None, None]) / (self.std[:, None, None] + 1e-6)
        else:
            ch_mean = inp.mean(axis=(1, 2), keepdims=True)
            ch_std = inp.std(axis=(1, 2), keepdims=True) + 1e-6
            inp = (inp - ch_mean) / ch_std
        inp = np.nan_to_num(inp, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
        with np.load(os.path.join(self.root, f'{fid}_pose.npz')) as pose_file:
            pose = pose_file['kp']
        kp = pose[0] if pose.ndim == 3 else pose.astype(np.float32)
        xy = kp[:, :2].astype(np.float32)
        vis = (kp[:, 2] > 0).astype(np.float32)
        scale_x = self.out_res / float(CAMERA_WIDTH)
        scale_y = self.out_res / float(CAMERA_HEIGHT)
        xy[:, 0] *= scale_x
        xy[:, 1] *= scale_y
        xy[:, 0] = np.clip(xy[:, 0], 0.0, self.out_res - 1)
        xy[:, 1] = np.clip(xy[:, 1], 0.0, self.out_res - 1)
        hms = self._make_heatmaps(xy, vis)
        return {
            'input': torch.from_numpy(inp),
            'target_hm': torch.from_numpy(hms.astype(np.float32)),
            'target_xy': torch.from_numpy(xy.astype(np.float32)),
            'vis': torch.from_numpy(vis.astype(np.float32)),
            'fid': fid,
        }


In [ ]:
# --- soft-argmax and model
class SoftArgmax2D(nn.Module):
    def __init__(self, beta=1.0):
        super().__init__()
        self.beta = beta
    def forward(self, hms):
        B, K, H, W = hms.shape
        h = torch.softmax(hms.view(B * K, -1) * self.beta, dim=-1).view(B * K, 1, H, W)
        xs = torch.linspace(0, W - 1, W, device=hms.device).view(1, 1, 1, W)
        ys = torch.linspace(0, H - 1, H, device=hms.device).view(1, 1, H, 1)
        x = torch.sum(h * xs, dim=[2, 3]).view(B, K, 1)
        y = torch.sum(h * ys, dim=[2, 3]).view(B, K, 1)
        coords = torch.cat([x, y], dim=-1)
        conf, _ = torch.max(h.view(B * K, -1), dim=-1)
        return coords, conf.view(B, K)

def _get_resnet(arch='resnet18', pretrained=True):
    arch = arch.lower()
    try:
        if arch == 'resnet18':
            base = torchvision.models.resnet18(weights=(torchvision.models.ResNet18_Weights.DEFAULT if pretrained else None))
        elif arch == 'resnet34':
            base = torchvision.models.resnet34(weights=(torchvision.models.ResNet34_Weights.DEFAULT if pretrained else None))
        elif arch == 'resnet50':
            base = torchvision.models.resnet50(weights=(torchvision.models.ResNet50_Weights.DEFAULT if pretrained else None))
        else:
            raise ValueError(f'Unsupported arch: {arch}')
    except Exception:
        # older torchvision fallback
        ctor = getattr(torchvision.models, arch)
        base = ctor(pretrained=pretrained)
    return base

class ResNetPoseNet(nn.Module):
    def __init__(self, num_kp=17, in_ch=2, out_res=128, arch='resnet18', pretrained=True, train_backbone=True):
        super().__init__()
        base = _get_resnet(arch, pretrained=pretrained)
        # adapt first conv to 2 channels if needed
        if in_ch != 3:
            old_conv = base.conv1
            new_conv = nn.Conv2d(in_ch, old_conv.out_channels,
                               kernel_size=old_conv.kernel_size,
                               stride=old_conv.stride,
                               padding=old_conv.padding,
                               bias=(old_conv.bias is not None))
            with torch.no_grad():
                if old_conv.weight.shape[1] == 3:
                    w = old_conv.weight.data
                    w_mean = w.mean(dim=1, keepdim=True)
                    new_w = w_mean.repeat(1, in_ch, 1, 1)
                    new_conv.weight.copy_(new_w)
                else:
                    nn.init.kaiming_normal_(new_conv.weight, mode='fan_out', nonlinearity='relu')
                if new_conv.bias is not None and old_conv.bias is not None:
                    new_conv.bias.copy_(old_conv.bias.data)
            base.conv1 = new_conv
        feat_channels = base.fc.in_features
        # backbone up to layer4 (remove avgpool & fc)
        self.backbone = nn.Sequential(*list(base.children())[:-2])
        if not train_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False
        # 5x upsample (4->8->16->32->64->128 for input 128)
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(feat_channels, 256, 4, 2, 1, bias=False), nn.BatchNorm2d(256), nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),         nn.BatchNorm2d(128), nn.ReLU(True),
            nn.ConvTranspose2d(128, 64,  4, 2, 1, bias=False),         nn.BatchNorm2d(64),  nn.ReLU(True),
            nn.ConvTranspose2d(64,  64,  4, 2, 1, bias=False),         nn.BatchNorm2d(64),  nn.ReLU(True),
            nn.ConvTranspose2d(64,  64,  4, 2, 1, bias=False),         nn.BatchNorm2d(64),  nn.ReLU(True),
        )
        self.hm_head = nn.Conv2d(64, num_kp, 1)
        self.soft = SoftArgmax2D(beta=10.0)
        self.out_res = out_res
    def forward(self, x):
        f = self.backbone(x)
        h = self.deconv(f)
        hm = self.hm_head(h)
        coords, conf = self.soft(hm)
        return hm, coords, conf


In [ ]:
# --- losses, metrics, training helpers
def heatmap_mse(pred, tgt, vis):
    w = vis.view(vis.size(0), vis.size(1), 1, 1)
    return F.mse_loss(pred * w, tgt * w, reduction='sum') / (w.sum() + 1e-6)

def coord_l1(pred, tgt, vis):
    w = vis.unsqueeze(-1)
    return F.l1_loss(pred * w, tgt * w, reduction='sum') / (w.sum() + 1e-6)

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval(); tot = 0; cnt = 0
    for b in loader:
        x = b['input'].to(device, non_blocking=True)
        y = b['target_xy'].to(device, non_blocking=True)
        vis = b['vis'].to(device, non_blocking=True)
        _, p, _ = model(x)
        w = vis.unsqueeze(-1)
        tot += torch.abs((p - y) * w).sum().item()
        cnt += w.sum().item()
    return tot / (cnt + 1e-6)

def _eval_pck(model, loader, device, alpha=0.05):
    model.eval(); correct = 0; total = 0
    H = W = getattr(model, 'out_res', 128)
    thresh = alpha * max(H, W)
    with torch.no_grad():
        for b in loader:
            x = b['input'].to(device, non_blocking=True)
            y = b['target_xy'].to(device, non_blocking=True)
            vis = b['vis'].to(device, non_blocking=True)
            _, p, _ = model(x)
            d = torch.norm((p - y), dim=-1)
            m = (vis > 0.5)
            correct += ((d < thresh) * m).sum().item()
            total += m.sum().item()
    return correct / max(total, 1)

def train_one_epoch(model, loader, opt, device, lam=0.5, epoch=None, scaler=None, use_amp=False, accum_steps=1):
    model.train(); loss_sum = 0.0
    pbar = tqdm(loader, total=len(loader), leave=False, desc=(f'Epoch {epoch:03d} [train]' if epoch is not None else 'Train'))
    opt.zero_grad(set_to_none=True)
    step_in_epoch = 0
    for step, b in enumerate(pbar, 1):
        x = b['input'].to(device, non_blocking=True)
        th = b['target_hm'].to(device, non_blocking=True)
        ty = b['target_xy'].to(device, non_blocking=True)
        vis = b['vis'].to(device, non_blocking=True)
        with amp.autocast(device_type='cuda', enabled=use_amp):
            ph, px, _ = model(x)
            loss_total = heatmap_mse(ph, th, vis) + lam * coord_l1(px, ty, vis)
        loss_value = float(loss_total.detach())
        loss_sum += loss_value * x.size(0)
        loss = loss_total / max(accum_steps, 1)
        if use_amp and scaler is not None:
            scaler.scale(loss).backward()
        else:
            loss.backward()
        if step % max(accum_steps, 1) == 0:
            if use_amp and scaler is not None:
                scaler.step(opt)
                scaler.update()
            else:
                opt.step()
            opt.zero_grad(set_to_none=True)
        step_in_epoch = step
        pbar.set_postfix(loss=loss_value)
    if step_in_epoch % max(accum_steps, 1) != 0:
        if use_amp and scaler is not None:
            scaler.step(opt)
            scaler.update()
        else:
            opt.step()
        opt.zero_grad(set_to_none=True)
    return loss_sum / max(len(loader.dataset), 1)


In [ ]:
# --- discover frames & split
files = sorted(glob.glob(os.path.join(DATA_DIR, '**', '*_radar.npz'), recursive=True))
ids = []
for f in files:
    rel = os.path.relpath(f, DATA_DIR)
    if rel.endswith('_radar.npz'):
        ids.append(rel[:-len('_radar.npz')])
random.seed(42)
random.shuffle(ids)
n_total = len(ids)
split_train = int(0.8 * n_total)
split_val = int(0.9 * n_total)
ids_tr = ids[:split_train]
ids_va = ids[split_train:split_val]
ids_te = ids[split_val:]
if len(ids_te) == 0 and len(ids_va) > 0:
    ids_te = ids_va[-1:]
    ids_va = ids_va[:-1]
elif len(ids_te) == 0 and len(ids_tr) > 1:
    ids_te = ids_tr[-1:]
    ids_tr = ids_tr[:-1]
print(f'{len(ids_tr)} train, {len(ids_va)} val, {len(ids_te)} test frames found.')


In [ ]:
# --- hyperparameters, loaders, model
BATCH = 128
EPOCHS = 120
LR = 3e-4
OUT_RES = 128
PATIENCE = 15
DEFAULT_WORKERS = int(os.environ.get('RADAR_NUM_WORKERS', 6))
if DEFAULT_WORKERS < 0:
    DEFAULT_WORKERS = 0
available_cpus = os.cpu_count() or DEFAULT_WORKERS or 1
if DEFAULT_WORKERS == 0:
    NUM_WORKERS = 0
else:
    NUM_WORKERS = min(DEFAULT_WORKERS, max(1, available_cpus - 2))
MP_CONTEXT = None
if NUM_WORKERS > 0:
    start_method = torch.multiprocessing.get_start_method(allow_none=True)
    if start_method is None:
        start_method = 'spawn' if sys.platform == 'win32' else 'fork'
    if start_method == 'spawn' and 'ipykernel' in sys.modules:
        print('Spawn start method detected in notebook; forcing DataLoader workers to 0.')
        NUM_WORKERS = 0
    elif start_method != 'fork':
        MP_CONTEXT = torch.multiprocessing.get_context(start_method)
ACCUM_STEPS = 1
USE_AMP = True
RESUME_FROM_LAST = True
LAST_CKPT_NAME = 'resnet_pose_last.pt'
BEST_CKPT_NAME = 'resnet_pose_best.pt'
stats_cache = os.path.join(OUT_DIR, f'radar_norm_res{OUT_RES}_log{int(LOG_SCALE_INPUT)}.npz')
mean, std = compute_channel_stats(DATA_DIR, ids_tr, OUT_RES, cache_path=stats_cache, log_scale=LOG_SCALE_INPUT)
print(f'Norm stats (mean={mean}, std={std})')
ds_tr = RadarPoseDataset(DATA_DIR, ids_tr, OUT_RES, mean=mean, std=std, log_scale=LOG_SCALE_INPUT)
ds_va = RadarPoseDataset(DATA_DIR, ids_va, OUT_RES, mean=mean, std=std, log_scale=LOG_SCALE_INPUT)
ds_te = RadarPoseDataset(DATA_DIR, ids_te, OUT_RES, mean=mean, std=std, log_scale=LOG_SCALE_INPUT)
loader_kwargs = dict(num_workers=NUM_WORKERS, pin_memory=True)
if NUM_WORKERS > 0:
    loader_kwargs.update(persistent_workers=True, prefetch_factor=2)
    if MP_CONTEXT is not None:
        loader_kwargs['multiprocessing_context'] = MP_CONTEXT
dl_tr = DataLoader(ds_tr, BATCH, shuffle=True, **loader_kwargs)
dl_va = DataLoader(ds_va, BATCH, shuffle=False, **loader_kwargs)
dl_te = DataLoader(ds_te, BATCH, shuffle=False, **loader_kwargs)
model = ResNetPoseNet(num_kp=17, in_ch=2, out_res=OUT_RES, arch='resnet18', pretrained=True, train_backbone=True).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=5, min_lr=1e-6)
BEST_CKPT_PATH = os.path.join(OUT_DIR, BEST_CKPT_NAME)
LAST_CKPT_PATH = os.path.join(OUT_DIR, LAST_CKPT_NAME)


In [ ]:
# --- train / resume / save
scaler = amp.GradScaler(enabled=USE_AMP)
H = W = getattr(model, 'out_res', 128)
best = float('inf')
no_improve = 0
start_epoch = 1
if RESUME_FROM_LAST and os.path.exists(LAST_CKPT_PATH):
    ckpt = load_checkpoint(LAST_CKPT_PATH, map_location=device, weights_only=False)
    try:
        model.load_state_dict(ckpt['model'])
        opt.load_state_dict(ckpt['optimizer'])
        sch.load_state_dict(ckpt['scheduler'])
        best = ckpt.get('best_val', best)
        no_improve = ckpt.get('no_improve', 0)
        start_epoch = ckpt.get('epoch', 1)
        scaler_state = ckpt.get('scaler')
        if USE_AMP and scaler_state:
            try:
                scaler.load_state_dict(scaler_state)
            except Exception as exc:
                print(f'Warning: could not load scaler state ({exc}); continuing without AMP state.')
        rng_state = ckpt.get('rng_state')
        if rng_state is not None:
            try:
                random.setstate(rng_state.get('python'))
                np.random.set_state(rng_state.get('numpy'))
                torch.set_rng_state(rng_state.get('torch'))
                cuda_state = rng_state.get('torch_cuda')
                if torch.cuda.is_available() and cuda_state is not None:
                    if isinstance(cuda_state, torch.Tensor):
                        cuda_state = [cuda_state]
                    restored_states = []
                    for state in cuda_state:
                        if state.is_cuda:
                            state = state.cpu()
                        restored_states.append(state)
                    torch.cuda.set_rng_state_all(restored_states)
            except Exception as exc:
                print(f'Warning: could not restore RNG state ({exc}); continuing without full RNG state.')
        print(f'Resuming from epoch {start_epoch} with best val {best:.3f} and no_improve={no_improve}')
    except Exception as exc:
        print(f'Warning: could not fully load checkpoint: {exc}')

for ep in range(start_epoch, EPOCHS + 1):
    tr = train_one_epoch(model, dl_tr, opt, device, epoch=ep, scaler=scaler if USE_AMP else None, use_amp=USE_AMP, accum_steps=ACCUM_STEPS)
    val = evaluate(model, dl_va, device)
    val_pck = _eval_pck(model, dl_va, device, alpha=0.05)
    sch.step(val)
    lr = opt.param_groups[0]['lr']
    print(f'Epoch {ep:03d} | lr {lr:.2e} | train {tr:.4f} | val MAE(px) {val:.3f} | PCK@0.05 {val_pck:.3f}')
    if val < best - 1e-3:
        best = val; no_improve = 0
        torch.save({'model': model.state_dict(), 'out_res': OUT_RES}, BEST_CKPT_PATH)
    else:
        no_improve += 1
    rng_state = {
        'python': random.getstate(),
        'numpy': np.random.get_state(),
        'torch': torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        cuda_states = torch.cuda.get_rng_state_all()
        rng_state['torch_cuda'] = [state.cpu().clone() for state in cuda_states]
    last_payload = {
        'epoch': ep + 1,
        'model': model.state_dict(),
        'optimizer': opt.state_dict(),
        'scheduler': sch.state_dict(),
        'scaler': scaler.state_dict() if USE_AMP else None,
        'best_val': best,
        'no_improve': no_improve,
        'out_res': OUT_RES,
        'rng_state': rng_state,
    }
    torch.save(last_payload, LAST_CKPT_PATH)
    if no_improve >= PATIENCE:
        print(f'Early stopping at epoch {ep} (no improvement {PATIENCE} epochs).')
        break
print('Best val MAE:', best)

# evaluate best on test
if os.path.exists(BEST_CKPT_PATH):
    ckpt = load_checkpoint(BEST_CKPT_PATH, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model']); model.eval()
    if len(ids_te) > 0:
        test_mae = evaluate(model, dl_te, device)
        test_pck = _eval_pck(model, dl_te, device, alpha=0.05)
        print(f'Test MAE(px) {test_mae:.3f} | PCK@0.05 {test_pck:.3f}')


In [ ]:
# --- visualization helpers
# COCO-style skeleton pairs for simple plotting
SKELETON = [
    (5, 7), (7, 9), (6, 8), (8, 10), (5, 6),
    (11, 13), (13, 15), (12, 14), (14, 16), (11, 12),
    (5, 11), (6, 12), (1, 2), (2, 3), (3, 4), (1, 5), (1, 6)
]
def _plot_pose(ax, coords, vis_mask, color, title, hw):
    H, W = hw
    ax.imshow(np.zeros((H, W)), cmap='gray', vmin=0, vmax=1)
    for (i, j) in SKELETON:
        if i-1 < coords.shape[0] and j-1 < coords.shape[0] and vis_mask[i-1] and vis_mask[j-1]:
            ax.plot([coords[i-1, 0], coords[j-1, 0]], [coords[i-1, 1], coords[j-1, 1]],
                    color=color, linewidth=2, alpha=0.9)
    ax.scatter(coords[vis_mask, 0], coords[vis_mask, 1], c=color, s=10)
    ax.set_title(title)
    ax.set_xlim(0, W); ax.set_ylim(H, 0)
    ax.set_aspect('equal')
def show_sample(sample, idx=0):
    hw = sample['input'].shape[1:]
    x = sample['input'].unsqueeze(0).to(device)
    with torch.no_grad():
        _, pred, _ = model(x)
    gt = sample['target_xy'].numpy()
    vis = sample['vis'].numpy() > 0.5
    fig, axes = plt.subplots(1, 2, figsize=(8, 4), constrained_layout=True)
    _plot_pose(axes[0], gt, vis, color='lime', title='Ground Truth', hw=hw)
    _plot_pose(axes[1], pred.squeeze(0).cpu().numpy(), vis, color='cyan', title='Prediction', hw=hw)
    fig.suptitle(f'Sample {idx}')
    plt.show()


In [ ]:
# Preview one validation sample overlay (optional)
# show_sample(ds_va[0], idx=0)
